# Pickwise — Conversational Shopping Assistant (Project P14)
### Product Search with MCP, RAG and Human Approval

**Team:** _names and roll numbers_

| Member | Contribution (sections, code, experiments) |
|---|---|
| | |
| | |
| | |

## 1. Problem Statement

- **Problem (our own words):** Shoppers describe a *situation* (“something to keep my kids busy at bath time that can get messy”), not keywords, and keyword search fails on such requests. We build an assistant that turns a long request into catalogue searches, recommends the top 3 products with reasons taken **only from the product text**, remembers each shopper's preferences across conversations, and **never adds to cart without the shopper's approval**.
- **Research question:** Does a multi-agent system with structured query rewriting recommend the gold product more often than plain hybrid retrieval on the raw request?
- **What success looks like:** Hit@3 / Recall@10 above the hybrid-on-raw-request baseline (target: _set after the retrieval baseline_); every recommendation reason supported by the product text; **100%** of `add_to_cart` actions approved by the shopper.

Code that is reused across the notebook, the MCP server and the app lives in the small `Pickwise/` package next to this notebook; each section below cites the course lab it builds on.

## 2. Setup

Builds on: Module 4 · Agentic_app (`config.py`, `llm.py`).

On Colab: clone the project repo first so `Pickwise/` and `requirements.txt` are next to the notebook, and install + start Ollama (`curl -fsSL https://ollama.com/install.sh | sh`, `ollama serve &`, `ollama pull qwen3:4b`).

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import random, time
import numpy as np, pandas as pd, torch
from datasets import load_dataset

from Pickwise import config
from Pickwise.data import build_catalogue, split_queries, check_no_leakage
from Pickwise.llm import get_llm, reasoning_trace
from Pickwise.retrieval import Retriever, tokenize
from Pickwise.metrics import evaluate
from Pickwise.need import ShoppingNeed, CATEGORY_GUIDE, analyse_many, to_need, search_query

SEED = config.SEED
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONFIG = {k: getattr(config, k) for k in [
    "MODEL", "REASONING", "EMBED_MODEL", "RERANK_MODEL", "DATASET_ID", "N_QUERIES", "DISTRACTOR_RATE",
    "N_DEV", "QUICK_RUN", "QUICK_N", "SEED", "RECURSION_LIMIT", "RISKY_TOOLS"]}
CONFIG["DEVICE"] = DEVICE
pd.Series(CONFIG, name="value").to_frame()

/home/additional/Projects/MCP-RAG-SHOPPPER/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,value
MODEL,qwen3:4b
REASONING,True
EMBED_MODEL,hyp1231/blair-roberta-base
RERANK_MODEL,Qwen/Qwen3-Reranker-0.6B
DATASET_ID,McAuley-Lab/Amazon-C4
N_QUERIES,1000
DISTRACTOR_RATE,0.03
N_DEV,200
QUICK_RUN,False
QUICK_N,100


**Model check.** `qwen3:4b` must be pulled in Ollama. We run it with `reasoning=True`: the reply in `content` is clean and the thinking goes to `additional_kwargs["reasoning_content"]`. With `reasoning=False`, Qwen3 still thinks but its narration leaks into the answer (measured in Phase 0; see `CURRENT_PROJECT_STATUS.md`).

In [3]:
import ollama
pulled = {m.model for m in ollama.list().models}
assert config.MODEL in pulled, f"run: ollama pull {config.MODEL}"

t = time.time()
reply = get_llm().invoke("In one sentence: what is a bath toy?")
print(f"{config.MODEL} answered in {time.time() - t:.1f}s")
print("answer   :", reply.content)
print("reasoning:", len(reasoning_trace(reply)), "chars (kept out of the answer)")

qwen3:4b answered in 3.0s
answer   : A bath toy is a safe, fun object designed specifically for babies and toddlers to play with during bath time.
reasoning: 1153 chars (kept out of the answer)


## 3. Data

Builds on: the brief §02 (dataset and sampling code).

| | |
|---|---|
| **Queries** | `McAuley-Lab/Amazon-C4`, `test` split (21,223 long requests, each with its gold `item_id`) → **fixed sample of 1,000**, `shuffle(seed=42)` |
| **Split** | first **200 = dev** (tuning only) · remaining **800 = test** (reported once) · `QUICK_RUN`: first 100 of test |
| **Catalogue** | `sampled_item_metadata_1M.jsonl` (1.06M items) → the 1,000 gold items **+ ~3% random distractors** (seed 42) |
| **Index text** | the item's `metadata` string (title + description). **`ori_review` is never used**: it is the review the query was written from (answer leakage) |

The catalogue is streamed with `load_dataset` on the first run and cached to `data/catalogue.parquet` (keyed on seed, rate and gold set), so re-runs skip the 640 MB scan.

In [4]:
queries = load_dataset(config.DATASET_ID)["test"].shuffle(seed=SEED).select(range(config.N_QUERIES))
print(queries)
words = pd.Series([len(q.split()) for q in queries["query"]])
print(f"query length: mean {words.mean():.0f} words, median {words.median():.0f}, max {words.max()}")
pd.DataFrame(queries.select(range(3)))[["qid", "query", "item_id"]]

Dataset({
    features: ['qid', 'query', 'item_id', 'user_id', 'ori_rating', 'ori_review'],
    num_rows: 1000
})
query length: mean 43 words, median 36, max 212


,qid,query,item_id
0,10853,I need a product that lasts longer than two we...,B0C6KQP8B3
1,20844,I need a thermometer that works great and has ...,B0BNM9KCF6
2,3197,I need candles that are perfect for weddings. ...,B09ZYBKLNS


In [5]:
t = time.time()
gold = set(queries["item_id"])
catalogue = build_catalogue(gold)
print(f"catalogue: {len(catalogue):,} items ({len(gold):,} gold + {len(catalogue) - len(gold):,} distractors) "
      f"in {time.time() - t:.0f}s")

missing = gold - set(catalogue["item_id"])
assert not missing, f"{len(missing)} gold items not in the catalogue"
check_no_leakage(queries, catalogue)
print("✓ every gold item is in the catalogue; ✓ no ori_review in the index")
catalogue.head(3)

catalogue: 32,614 items (998 gold + 31,616 distractors) in 0s
✓ every gold item is in the catalogue; ✓ no ori_review in the index


,item_id,category,title,text
0,B07NRD63N7,Clothing,Skirt Sports Women's Kelly Bra,Skirt Sports Women's Kelly Bra. Skirt Sports -...
1,B005EDIEDI,Toys,"Axial AXA1202 O-Ring, 3.5x2mm","Axial AXA1202 O-Ring, 3.5x2mm. These are the A..."
2,B08G4XMSV4,Electronics,Beeyoka Privacy Screen Protector Compatible wi...,Beeyoka Privacy Screen Protector Compatible wi...


In [6]:
dev, test = split_queries(queries)
print(f"dev: {len(dev)} queries · test: {len(test)} queries (QUICK_RUN={config.QUICK_RUN})")

lengths = catalogue["text"].str.split().str.len()
print(f"item text: mean {lengths.mean():.0f} words, median {lengths.median():.0f}, max {lengths.max():,}")
catalogue["category"].value_counts().rename("items").to_frame().T

dev: 200 queries · test: 800 queries (QUICK_RUN=False)


item text: mean 86 words, median 32, max 12,237


category,Home,Clothing,Electronics,Tools,Household,Care,Kindle,Pet,Automotive,Garden,...,Instruments,Appliances,Handmade,CDs,Fashion,Health,Beauty,Software,Gift,Magazine
items,5251,4445,2420,2219,1994,1928,1705,1404,1345,1274,...,205,151,61,59,56,24,23,20,8,4


**Data quality.** Some catalogue items have little or no text (e.g. `"Her Leaves Remain. ."`). Queries whose gold item is empty are **unanswerable** by any retriever. We keep them in the test set (dropping them would flatter the numbers) and report them here and in the error analysis.

In [7]:
words = catalogue.set_index("item_id")["text"].str.split().str.len()
gold_words = words.loc[list(gold)]
print(f"gold items: {len(gold)} (2 queries share a gold item)")
print(f"gold items with empty text: {(gold_words == 0).sum()} · with ≤ 5 words: {(gold_words <= 5).sum()}")
print(f"test queries with an empty gold item: {sum(words[i] == 0 for i in test['item_id'])} of {len(test)}")

gold items: 998 (2 queries share a gold item)
gold items with empty text: 4 · with ≤ 5 words: 15
test queries with an empty gold item: 4 of 800


## 4. Baseline

Builds on: Module 3 · RAG_Pipeline_Lab (BM25, FAISS dense retrieval, Recall@k / MRR) and Module 4 · Agentic_app (`rag.py`, the lab pipeline as a module) → `Pickwise/retrieval.py`, `Pickwise/metrics.py`.

**Index: one document per product**, its full `metadata` text. The median product has only 32 words, so chunking would mostly give one chunk per product anyway, and it would need a chunk → product merge before scoring. Chunking is kept as a comparison row in §5a.

**Baselines (from the brief):** BM25 and dense retrieval on the **raw** request.

**Metrics** against the single gold item, on the fixed test subset: Recall@10, MRR (over the top 100) and nDCG@10. Recall@50/100 show what a re-ranker could still recover. With 800 test queries the standard error on Recall@10 is about ±1.8 points (±3.5 on the 200 dev queries).

**Changes from the lab for 32k products:** queries are embedded and re-ranked in batches; document embeddings are cached in `data/`; and BM25 uses the lab's `rank_bm25` formula as one sparse-matrix product. That gives identical scores (checked below) and is ~600× faster.

In [8]:
t = time.time()
# Phases 2-3 run the Module 3 lab models (the brief's defaults); the upgrade at the end of §5b swaps them
retriever = Retriever(catalogue, embed_model=config.LAB_EMBED_MODEL, rerank_model=config.LAB_RERANK_MODEL,
                    rerank_top_n=config.LAB_RERANK_TOP_N)   # BM25 + FAISS (bge-small) · ms-marco re-ranker
print(f"index: {len(retriever.texts):,} documents in {time.time() - t:.0f}s on {DEVICE}")

from rank_bm25 import BM25Okapi           # the lab's BM25, to check the vectorised copy
lab_bm25 = BM25Okapi([tokenize(text) for text in retriever.texts])
for q in dev["query"][:3]:
    assert np.allclose(lab_bm25.get_scores(tokenize(q)), retriever.bm25.get_scores(tokenize(q)))
print("✓ vectorised BM25 gives the same scores as rank_bm25")

index: 32,614 documents in 1s on cuda


✓ vectorised BM25 gives the same scores as rank_bm25


In [9]:
results = []   # one row per condition, all on the same fixed test subset -> §6
ranks = {}     # condition -> per-query rank of the gold item (for paired comparisons)

def report(condition, queries, method, r=None, **kw):
    """Score one condition on the test subset and add it to the results table."""
    metrics, ranks[condition] = evaluate(r or retriever, list(queries), list(test["item_id"]), method,
                                         return_ranks=True, **kw)
    results.append({"condition": condition, "n": len(test), **metrics})
    return metrics

raw_test = list(test["query"])
report("BM25 · raw request", raw_test, "bm25")
report("Dense · raw request", raw_test, "dense")
pd.DataFrame(results).set_index("condition").round(3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16586.18it/s]

,n,Recall@10,MRR,nDCG@10,Recall@50,Recall@100
condition,,,,,,
BM25 · raw request,800,0.098,0.062,0.068,0.148,0.200
Dense · raw request,800,0.409,0.248,0.279,0.592,0.672


## 5. System

### 5a. Retrieval / RAG

Builds on: Module 3 · RAG_Pipeline_Lab (hybrid RRF, cross-encoder re-ranking).

**Hybrid** = Reciprocal Rank Fusion of BM25 and dense (k = 60). **Re-rank** = the cross-encoder `ms-marco-MiniLM-L-6-v2` re-scores the hybrid top 50. First, a small sweep **on dev** (test is never used for choices):

In [10]:
raw_dev, gold_dev = list(dev["query"]), list(dev["item_id"])

def dev_row(setting, queries=None, r=None, **kw):
    return {"setting": setting, **evaluate(r or retriever, queries or raw_dev, gold_dev, **kw)}

no_instruction = Retriever(catalogue, query_prefix=False, embed_model=config.LAB_EMBED_MODEL, rerank_model=config.LAB_RERANK_MODEL,
                    rerank_top_n=config.LAB_RERANK_TOP_N)
dev_sweep = pd.DataFrame([
    dev_row("dense, bge query instruction", method="dense"),
    dev_row("dense, no instruction", r=no_instruction, method="dense"),
    *[dev_row(f"hybrid, BM25 weight {w}", method="hybrid", weights=(w, 1.0)) for w in (1.0, 0.5, 0.25)],
    *[dev_row(f"hybrid + re-rank top {n}", method="hybrid+rerank", top_n=n) for n in (20, 50, 100)],
]).set_index("setting")
dev_sweep.round(3)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 18280.77it/s]

,Recall@10,MRR,nDCG@10,Recall@50,Recall@100
setting,,,,,
"dense, bge query instruction",0.465,0.279,0.318,0.595,0.655
"dense, no instruction",0.445,0.276,0.310,0.600,0.640
"hybrid, BM25 weight 1.0",0.325,0.159,0.188,0.550,0.615
"hybrid, BM25 weight 0.5",0.420,0.208,0.250,0.600,0.650
"hybrid, BM25 weight 0.25",0.455,0.224,0.272,0.600,0.655
hybrid + re-rank top 20,0.460,0.246,0.294,0.550,0.615
hybrid + re-rank top 50,0.470,0.240,0.290,0.550,0.615
hybrid + re-rank top 100,0.435,0.225,0.268,0.610,0.615


**On dev:** BM25 collapses on these 43-word situational requests, which is the brief's premise. With equal weights, RRF lets it drag hybrid below dense. Down-weighting BM25 moves hybrid back towards dense. The re-ranker, trained on short web questions, recovers Recall@10 but not MRR. The bge query instruction helps a little. Most differences sit inside the ±3.5-point dev noise, so we keep the **lab defaults** (equal weights, re-rank top 50) with the bge instruction, and score the brief's four variants on test:

In [11]:
report("Hybrid (RRF) · raw request", raw_test, "hybrid")
report("Hybrid + re-rank · raw request", raw_test, "hybrid+rerank")
pd.DataFrame(results).set_index("condition").round(3)

,n,Recall@10,MRR,nDCG@10,Recall@50,Recall@100
condition,,,,,,
BM25 · raw request,800,0.098,0.062,0.068,0.148,0.200
Dense · raw request,800,0.409,0.248,0.279,0.592,0.672
Hybrid (RRF) · raw request,800,0.315,0.158,0.186,0.528,0.602
Hybrid + re-rank · raw request,800,0.405,0.232,0.267,0.528,0.602


**Chunking comparison.** This uses the Module 3 lab setup: 300-character chunks with 50 overlap, each prefixed with the product title as a contextual header. A product's score is its best chunk's score.

In [12]:
t = time.time()
chunked = Retriever(catalogue, chunked=True, embed_model=config.LAB_EMBED_MODEL, rerank_model=config.LAB_RERANK_MODEL,
                    rerank_top_n=config.LAB_RERANK_TOP_N)
print(f"chunked index: {len(chunked.texts):,} chunks for {len(catalogue):,} products ({time.time() - t:.0f}s)")
report("Hybrid + re-rank · raw request · chunked index", raw_test, "hybrid+rerank", r=chunked)
pd.DataFrame(results).set_index("condition").round(3).tail(2)

chunked index: 83,021 chunks for 32,614 products (2s)


,n,Recall@10,MRR,nDCG@10,Recall@50,Recall@100
condition,,,,,,
Hybrid + re-rank · raw request,800,0.405,0.232,0.267,0.528,0.602
Hybrid + re-rank · raw request · chunked index,800,0.399,0.226,0.261,0.509,0.581


**Test, raw request:**
- **Dense retrieval is the strongest single retriever** (Recall@10 ≈ 0.41, MRR ≈ 0.25). **BM25 finds the gold item in the top 10 for only about 1 request in 10.**
- Equal-weight hybrid sits between the two, and the re-ranker lifts it back to dense level.
- The **chunked index scores the same** as one document per product with 2.5× more entries, so the system keeps one document per product.
- About a third of gold items are in no top 100. On raw requests, the general-purpose embedder is the bottleneck, which makes the product-specific BLaIR embedder (stretch goal) the most promising upgrade.

### 5b. Agents and tools

#### Need analyst (Phase 3)

Builds on: Module 2 · Structured_Generation_Lab (schema-constrained generation, with `Literal` choices compiled into the generation grammar) → `Pickwise/need.py`.

The need analyst turns the long request into `{product_type, must_haves, category}` with `with_structured_output`. `category` is a `Literal` over the catalogue's 31 categories. The short names are ambiguous, so the prompt explains what each one holds (e.g. *Care* = beauty & personal care, *Household* = health & household). Replies are cached in `data/needs/`, so re-runs don't call the model again.

**Reasoning mode.** Free-text answers and tool calls need `reasoning=True`; otherwise Qwen3's narration leaks into the reply (§2). For this schema-constrained call we use `reasoning=False`: the schema keeps the reply clean either way, and it is much faster. This is checked on dev below.

**Searching with the rewrite.** The searcher's query is the rewrite followed by the shopper's own words (`search_query(need, request)`). If the analyst fails on a request, the search falls back to the raw request.

In [13]:
assert set(CATEGORY_GUIDE) == set(catalogue["category"]), "schema categories must match the catalogue"
gold_category = dict(zip(catalogue["item_id"], catalogue["category"]))
gold_title = dict(zip(catalogue["item_id"], catalogue["title"]))

def summarise(records, split):
    needs = [to_need(records[q]) for q in split["qid"]]
    seconds = np.array([records[q]["seconds"] for q in split["qid"]])
    accuracy = np.mean([n is not None and n.category == gold_category[g] for n, g in zip(needs, split["item_id"])])
    print(f"{len(needs)} requests · {sum(n is None for n in needs)} failures · "
          f"{seconds.mean():.2f}s mean per request · category accuracy {accuracy:.1%}")
    return needs

print("dev: ", end=""); need_dev = summarise(analyse_many(dev["qid"], dev["query"]), dev)
print("test:", end=" "); need_test = summarise(analyse_many(test["qid"], test["query"]), test)

for row, n in list(zip(dev, need_dev))[:3]:
    print(f"\nREQUEST  {row['query'][:160]}")
    print(f"REWRITE  {search_query(n)}  [{n.category}]")
    print(f"GOLD     {gold_title[row['item_id']][:110]}  [{gold_category[row['item_id']]}]")

dev: 200 requests · 0 failures · 1.06s mean per request · category accuracy 43.0%
test: 800 requests · 0 failures · 1.09s mean per request · category accuracy 51.9%

REQUEST  I need a product that lasts longer than two weeks and is effective in removing clusters. It's important that it's easy to clean when I take it off.
REWRITE  air purifier filter, lasts longer than two weeks, removes clusters, easy to clean when taken off  [Electronics]
GOLD     Lash Bond and Seal, CALAILIS Cluster Lash Glue for Lash Clusters Super Strong Hold and Long Lasting 48-72 Hour  [Care]

REQUEST  I need a thermometer that works great and has an auto off feature. I want one that uses regular batteries instead of a large watch battery. It should also turn 
REWRITE  thermometer with auto off, works great, auto off feature, uses regular batteries, turns on and off automatically when probe extended, no dirty probe  [Electronics]
GOLD     ThermoPro TP19H Waterproof Digital Meat Thermometer for Grilling with Ambid

In [14]:
# reasoning on vs off for the need analyst, on dev (~30 s per request the first time with reasoning on)
from Pickwise.metrics import paired_bootstrap
check = dev.select(range(30 if config.QUICK_RUN else len(dev)))
compare, check_ranks = [], {}
for reasoning in (False, True):
    records = analyse_many(check["qid"], check["query"], reasoning=reasoning)
    needs = [to_need(records[q]) for q in check["qid"]]
    queries = [search_query(n, r) if n else r for n, r in zip(needs, check["query"])]
    metrics, check_ranks[reasoning] = evaluate(retriever, queries, list(check["item_id"]), "hybrid+rerank",
                                               return_ranks=True)
    compare.append({
        "reasoning": reasoning,
        "s / request": np.mean([records[q]["seconds"] for q in check["qid"]]),
        "failures": sum(n is None for n in needs),
        "category acc.": np.mean([n is not None and n.category == gold_category[g] for n, g in zip(needs, check["item_id"])]),
        **metrics,
    })
display(pd.DataFrame(compare).set_index("reasoning").round(3))
print("reasoning on − off:", {k: (tuple(round(x, 3) for x in v) if isinstance(v, tuple) else round(v, 3))
                              for k, v in paired_bootstrap(check_ranks[True], check_ranks[False]).items()})

,s / request,failures,category acc.,Recall@10,MRR,nDCG@10,Recall@50,Recall@100
reasoning,,,,,,,,
False,1.061,0,0.430,0.475,0.260,0.305,0.585,0.660
True,30.652,0,0.525,0.465,0.276,0.313,0.590,0.675


reasoning on − off: {'Δ Recall@10': np.float64(-0.01), 'Recall@10 95% CI': (np.float64(-0.045), np.float64(0.025)), 'Δ MRR': np.float64(0.015), 'MRR 95% CI': (np.float64(-0.009), np.float64(0.041))}


**On dev: reasoning off is confirmed.**
- With reasoning on, each request takes **~28× longer** (≈ 31 s vs ≈ 1 s), with **no measurable retrieval gain**: the paired bootstrap intervals for Recall@10 and MRR both include zero.
- Thinking first does pick the gold item's category more often (≈ 52% vs ≈ 43%). Category isn't used as a filter, so that doesn't reach the search. We note it in case category is later used as a soft boost.
- Neither mode produced an invalid reply: the schema keeps the output clean either way.

The need analyst therefore runs with `reasoning=False`.

**Does rewriting beat the raw request?** On dev, we compare three query forms across the four retrievers, plus the predicted category used as a hard filter:

In [15]:
forms = {
    "raw request": raw_dev,
    "rewrite": [search_query(n) if n else r for n, r in zip(need_dev, raw_dev)],
    "rewrite + request": [search_query(n, r) if n else r for n, r in zip(need_dev, raw_dev)],
}
grid = pd.DataFrame([{"query": f, "method": m, **evaluate(retriever, qs, gold_dev, m)}
                     for f, qs in forms.items() for m in ("bm25", "dense", "hybrid", "hybrid+rerank")])
display(grid.pivot(index="method", columns="query", values=["Recall@10", "MRR"]).round(3))

predicted = [n.category if n else None for n in need_dev]
pd.DataFrame([dev_row(f"hybrid + re-rank · {f} · {'category filter' if c else 'no filter'}",
                      queries=forms[f], method="hybrid+rerank", categories=c)
              for f in ("raw request", "rewrite + request") for c in (None, predicted)]).set_index("setting").round(3)

Recall@10                                   MRR          \
query         raw request rewrite rewrite + request raw request rewrite   
method                                                                    
bm25                0.070   0.310             0.215       0.050   0.180   
dense               0.465   0.420             0.445       0.279   0.260   
hybrid              0.325   0.385             0.395       0.159   0.236   
hybrid+rerank       0.470   0.455             0.475       0.240   0.243   

                                 
query         rewrite + request  
method                           
bm25                      0.124  
dense                     0.277  
hybrid                    0.216  
hybrid+rerank             0.260

,Recall@10,MRR,nDCG@10,Recall@50,Recall@100
setting,,,,,
hybrid + re-rank · raw request · no filter,0.470,0.240,0.290,0.550,0.615
hybrid + re-rank · raw request · category filter,0.225,0.122,0.143,0.285,0.330
hybrid + re-rank · rewrite + request · no filter,0.475,0.260,0.305,0.585,0.660
hybrid + re-rank · rewrite + request · category filter,0.230,0.137,0.157,0.275,0.365


**On dev:**
- **Rewriting rescues BM25** (Recall@10 0.07 → 0.31) and hybrid. Dense retrieval on the raw request is already about as good as any rewrite at the top 10, and every setting is within the ±3.5-point dev noise.
- **Rewrite + request** works well across retrievers: the rewrite gives BM25 clean keywords, and the shopper's own words keep what dense retrieval and the re-ranker need.
- **The category filter halves recall.** The analyst picks the gold item's category only about 43% of the time, because the short names overlap (e.g. Home / Tools / Household), and a wrong filter removes the gold item outright.

**Chosen on dev:** hybrid + re-rank on *rewrite + request*, **no category filter**. `category` stays an optional tool argument for when the shopper names one. We also tried giving each retriever its own query form (BM25 on the rewrite, dense on the raw request, fused with RRF); it was no better than this single query.

In [16]:
rewrite_test = [search_query(n) if n else r for n, r in zip(need_test, raw_test)]
system_test = [search_query(n, r) if n else r for n, r in zip(need_test, raw_test)]
report("BM25 · rewrite + request", system_test, "bm25")
report("Dense · rewrite + request", system_test, "dense")
report("Hybrid (RRF) · rewrite + request", system_test, "hybrid")
report("Hybrid + re-rank · rewrite only", rewrite_test, "hybrid+rerank")
report("Hybrid + re-rank · rewrite + request", system_test, "hybrid+rerank")
pd.DataFrame(results).set_index("condition").round(3)

,n,Recall@10,MRR,nDCG@10,Recall@50,Recall@100
condition,,,,,,
BM25 · raw request,800,0.098,0.062,0.068,0.148,0.200
Dense · raw request,800,0.409,0.248,0.279,0.592,0.672
Hybrid (RRF) · raw request,800,0.315,0.158,0.186,0.528,0.602
Hybrid + re-rank · raw request,800,0.405,0.232,0.267,0.528,0.602
Hybrid + re-rank · raw request · chunked index,800,0.399,0.226,0.261,0.509,0.581
BM25 · rewrite + request,800,0.234,0.140,0.158,0.328,0.381
Dense · rewrite + request,800,0.422,0.247,0.282,0.596,0.664
Hybrid (RRF) · rewrite + request,800,0.361,0.217,0.243,0.560,0.636
Hybrid + re-rank · rewrite only,800,0.442,0.257,0.294,0.605,0.682


In [17]:
from Pickwise.metrics import paired_bootstrap

# Same 800 queries under both conditions -> paired bootstrap (2,000 resamples of the queries)
rewritten = "Hybrid + re-rank · rewrite + request"
pd.DataFrame({f"rewritten − {base}": paired_bootstrap(ranks[rewritten], ranks[base])
              for base in ("Dense · raw request", "Hybrid + re-rank · raw request")}).T.map(
    lambda v: f"[{v[0]:+.3f}, {v[1]:+.3f}]" if isinstance(v, tuple) else f"{v:+.3f}")

,Δ Recall@10,Recall@10 95% CI,Δ MRR,MRR 95% CI
rewritten − Dense · raw request,+0.044,"[+0.019, +0.069]",+0.027,"[+0.007, +0.048]"
rewritten − Hybrid + re-rank · raw request,+0.048,"[+0.025, +0.072]",+0.043,"[+0.027, +0.061]"


**Test: does rewriting beat the raw request? Yes, for the full pipeline.**
- Hybrid + re-rank on *rewrite + request* reaches **Recall@10 ≈ 0.45, MRR ≈ 0.28**. On the same 800 queries, the paired bootstrap above puts it ahead of both the raw-request pipeline and the best baseline (dense on the raw request): the 95% intervals for Recall@10 and MRR exclude zero.
- Rewriting more than doubles BM25 and lifts hybrid, while dense alone barely moves. **The gain comes from giving the keyword side and the re-ranker a clean query.**
- Rewrite-only lands in between, so keeping the shopper's words helps.

Next, the two retrieval models themselves are upgraded.

#### Retrieval model upgrade (stretch goal: product-specific embedder)

Builds on: the brief's optional product embedder (`hyp1231/blair-roberta-base`, BLaIR), and its rule that other open models are fine if justified.

With the lab models, about a third of gold items are in no top 100, so the **embedder is the bottleneck**. The ms-marco re-ranker, trained on short web questions, barely moves MRR. We compare alternatives **on dev**, using the *rewrite + request* query:
- **Embedders:** `bge-small-en-v1.5` (lab); **BLaIR** (`blair-roberta-base`, 0.1B, pretrained on Amazon Reviews 2023 to match item text with how people talk about items, with CLS pooling as on its model card); and `Qwen3-Embedding-0.6B` (a strong general-purpose embedder).
- **Re-rankers:** `ms-marco-MiniLM-L-6-v2` (lab), `bge-reranker-base`, `bge-reranker-v2-m3`, and **`Qwen3-Reranker-0.6B`** (a Qwen model under 4B, as the brief allows).

EmbeddingGemma-300M is gated on Hugging Face and was not tested. The first run embeds the catalogue with each new model (≈ 2 min for BLaIR, ≈ 5 min for Qwen3-Embedding on a GPU); the vectors are cached.

In [18]:
BLAIR, QWEN_EMB, QWEN_RR = "hyp1231/blair-roberta-base", "Qwen/Qwen3-Embedding-0.6B", "Qwen/Qwen3-Reranker-0.6B"
system_dev = forms["rewrite + request"]
by_embedder = {name: Retriever(catalogue, embed_model=name) for name in (config.LAB_EMBED_MODEL, BLAIR, QWEN_EMB)}

upgrade = [dev_row("lab models · hybrid + re-rank (ms-marco, top 50)", queries=system_dev, method="hybrid+rerank")]
for name, r in by_embedder.items():
    upgrade.append(dev_row(f"{name.split('/')[-1]} · dense · raw request", r=r, method="dense"))
    upgrade.append(dev_row(f"{name.split('/')[-1]} · dense", r=r, queries=system_dev, method="dense"))
upgrade.append(dev_row("bge-small · hybrid + re-rank (Qwen3-Reranker, top 20)", r=by_embedder[config.LAB_EMBED_MODEL],
                       queries=system_dev, method="hybrid+rerank"))
blair = by_embedder[BLAIR]
for reranker in (config.LAB_RERANK_MODEL, "BAAI/bge-reranker-base", "BAAI/bge-reranker-v2-m3", QWEN_RR):
    blair.rerank_model = reranker
    upgrade.append(dev_row(f"BLaIR · hybrid + re-rank ({reranker.split('/')[-1]}, top 20)", r=blair,
                           queries=system_dev, method="hybrid+rerank"))
for n in (10, 30, 50):
    upgrade.append(dev_row(f"BLaIR · hybrid + re-rank (Qwen3-Reranker, top {n})", r=blair,
                           queries=system_dev, method="hybrid+rerank", top_n=n))
pd.DataFrame(upgrade).set_index("setting").round(3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18756.97it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 20462.27it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 18191.02it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 17434.09it/s]

,Recall@10,MRR,nDCG@10,Recall@50,Recall@100
setting,,,,,
"lab models · hybrid + re-rank (ms-marco, top 50)",0.475,0.260,0.305,0.585,0.660
bge-small-en-v1.5 · dense · raw request,0.465,0.279,0.318,0.595,0.655
bge-small-en-v1.5 · dense,0.445,0.277,0.309,0.615,0.670
blair-roberta-base · dense · raw request,0.610,0.364,0.417,0.760,0.795
blair-roberta-base · dense,0.580,0.341,0.391,0.730,0.775
Qwen3-Embedding-0.6B · dense · raw request,0.465,0.302,0.335,0.625,0.710
Qwen3-Embedding-0.6B · dense,0.485,0.314,0.349,0.645,0.695
"bge-small · hybrid + re-rank (Qwen3-Reranker, top 20)",0.480,0.335,0.366,0.585,0.660
"BLaIR · hybrid + re-rank (ms-marco-MiniLM-L-6-v2, top 20)",0.580,0.300,0.361,0.715,0.750


**On dev:**
- **BLaIR is the biggest single gain in the project.** Dense retrieval on the raw request goes from Recall@10 0.47 to 0.61, and Recall@100 from 0.66 to 0.80. The general-purpose Qwen3-Embedding, 6× larger, gains little: a product-specific embedder beats a bigger general one.
- **Qwen3-Reranker-0.6B is the only re-ranker that clearly improves MRR.** On BLaIR's hybrid list it reaches 0.38, against 0.30 with ms-marco and 0.34 with bge-reranker-v2-m3. It also helps on top of bge-small.
- **Re-ranking the top 20 is as good as the top 50 and 2.5× faster**; the top 10 is too shallow.

**Chosen on dev:** BLaIR · hybrid RRF · Qwen3-Reranker on the top 20, with the *rewrite + request* query. It is the best on MRR and nDCG@10, and it keeps the hybrid + re-ranker pipeline the brief asks for. Scored once on test:

In [19]:
system_retriever = Retriever(catalogue)     # config defaults: BLaIR · hybrid RRF · Qwen3-Reranker top 20
report("BLaIR · dense · raw request", raw_test, "dense", r=system_retriever)
report("bge-small · hybrid + Qwen3 re-rank · rewrite + request", system_test, "hybrid+rerank",
       r=by_embedder[config.LAB_EMBED_MODEL])
report("BLaIR · hybrid + ms-marco re-rank · rewrite + request", system_test, "hybrid+rerank",
       r=Retriever(catalogue, rerank_model=config.LAB_RERANK_MODEL, rerank_top_n=config.LAB_RERANK_TOP_N))
report("BLaIR · hybrid + Qwen3 re-rank · raw request", raw_test, "hybrid+rerank", r=system_retriever)
SYSTEM = "BLaIR · hybrid + Qwen3 re-rank · rewrite + request  ← system retriever"
report(SYSTEM, system_test, "hybrid+rerank", r=system_retriever)
pd.DataFrame(results).set_index("condition").round(3).tail(6)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 22677.75it/s]

,n,Recall@10,MRR,nDCG@10,Recall@50,Recall@100
condition,,,,,,
Hybrid + re-rank · rewrite + request,800,0.452,0.276,0.312,0.560,0.636
BLaIR · dense · raw request,800,0.575,0.356,0.401,0.741,0.804
bge-small · hybrid + Qwen3 re-rank · rewrite + request,800,0.430,0.303,0.329,0.560,0.636
BLaIR · hybrid + ms-marco re-rank · rewrite + request,800,0.504,0.300,0.339,0.695,0.758
BLaIR · hybrid + Qwen3 re-rank · raw request,800,0.568,0.360,0.406,0.685,0.752
BLaIR · hybrid + Qwen3 re-rank · rewrite + request ← system retriever,800,0.548,0.360,0.400,0.695,0.758


In [20]:
pd.DataFrame({f"system − {base}": paired_bootstrap(ranks[SYSTEM], ranks[base])
              for base in ("Dense · raw request", "Hybrid + re-rank · rewrite + request",
                           "BLaIR · dense · raw request", "BLaIR · hybrid + Qwen3 re-rank · raw request")}).T.map(
    lambda v: f"[{v[0]:+.3f}, {v[1]:+.3f}]" if isinstance(v, tuple) else f"{v:+.3f}")

,Δ Recall@10,Recall@10 95% CI,Δ MRR,MRR 95% CI
system − Dense · raw request,+0.139,"[+0.107, +0.170]",+0.112,"[+0.090, +0.135]"
system − Hybrid + re-rank · rewrite + request,+0.095,"[+0.066, +0.124]",+0.085,"[+0.063, +0.106]"
system − BLaIR · dense · raw request,-0.028,"[-0.054, +0.000]",+0.004,"[-0.018, +0.027]"
system − BLaIR · hybrid + Qwen3 re-rank · raw request,-0.020,"[-0.044, +0.005]",+0.000,"[-0.015, +0.016]"


**Test: the model upgrade is a large, real gain.**
- The system retriever reaches **Recall@10 ≈ 0.55, MRR ≈ 0.36, nDCG@10 ≈ 0.40**: about +9.5 points Recall@10 and +0.085 MRR over the same pipeline with the lab models, and +14 points Recall@10 over the best baseline (dense retrieval with bge-small on the raw request). All paired 95% intervals exclude zero.
- Both swaps contribute. Qwen3-Reranker alone lifts bge-small's MRR (0.28 → 0.30). BLaIR with the old ms-marco re-ranker reaches Recall@10 ≈ 0.50.

**With BLaIR, the rest of the pipeline adds nothing measurable.** BLaIR's dense search on the raw request alone is level with the full system (MRR difference ≈ 0, interval includes zero; Recall@10 −3 points, borderline). Rewriting no longer helps either, because BLaIR was pretrained on exactly this kind of review-style language. On dev the full pipeline led on MRR; test was scored once and isn't used for choices, so the system keeps the brief's hybrid + re-ranker over MCP. Two consequences:
1. **A much simpler and faster retriever, BLaIR on the raw request (~5 ms and no re-ranker, also fast on CPU), is an equally good fallback.** This matters for laptop-CPU runs.
2. BLaIR was pretrained on Amazon Reviews 2023, which may include the reviews the C4 queries were written from. The lab-model rows show the gains without BLaIR, and this goes in **Limitations**.

The need analyst is still the step that understands the request for the rest of the conversation: refinements ("something cheaper"), preferences, and what the comparator is choosing for.

#### Searcher, comparator, MCP server, memory and approval

_Phases 4–5 · MCP server + local `@tool`s, LangGraph supervisor, memory, `interrupt()` before `add_to_cart` (Module 4 · Multi_Agent_System_End_to_End, MCP_Server_and_Tools_Lab)._

### 5c. Fine-tuning (optional)

_Phase 8 stretch · LoRA need analyst._

## 6. Evaluation

_Phase 7 · one results table: baselines, full system, ablations — same test subset._

## 7. Error Analysis

_Phase 7 · at least 10 categorised failures._

## 8. Demo

_Phase 6 · multi-turn chat with live trace; pauses at the cart for approve / reject._

## 9. Limitations & Ethics

- Where does the system fail?
- Who could be harmed if it is wrong?
- Privacy, bias and misuse considerations.

## 10. References

- Dataset: McAuley-Lab/Amazon-C4 (Hou et al., 2024, *Bridging Language and Items for Retrieval and Recommendation*)
- Models: qwen3:4b (Ollama), BAAI/bge-small-en-v1.5, cross-encoder/ms-marco-MiniLM-L-6-v2
- Course labs reused: see each section
- Tools and libraries: LangGraph, LangChain, MCP, FAISS, rank-bm25, sentence-transformers, Hugging Face datasets